# Top-3 GPU benchmark (Swin-T / ConvNeXt-T / DenseNet-121)

Set the runtime to **GPU** before running. Push the updated `main` branch first, or replace `REPO_URL` with an accessible copy of this repository.

Runs **one selected model** at a time with `SEEDS=42,43,44` on the P-disjoint splits. Set `ACTIVE_RUN_NAME` in the config cell to choose the model. The dataset must contain `ResearchImage/`, `Images_v1/`, `Images_v2/` under one parent. On Colab put them in Google Drive; on Kaggle attach them as a dataset. The notebook links that parent to `data/dataset`.


In [ ]:
import os
import sys
import shutil
import pathlib
import subprocess

COLAB = "google.colab" in sys.modules
KAGGLE = os.path.exists("/kaggle/input")
print("environment:", "colab" if COLAB else "kaggle" if KAGGLE else "local")

In [ ]:
REPO_URL = "https://github.com/HasanImamB/SkinDiseaseBD.git"
WORKDIR = "/content/SkinDiseaseBD" if COLAB else "/kaggle/working/SkinDiseaseBD"
repo = pathlib.Path(WORKDIR)
if repo.exists():
    subprocess.run(["git", "-C", WORKDIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, WORKDIR], check=True)
print(repo)

In [ ]:
# Patch train.py for DINOv2/img_size compatibility (safe no-op for these models)
train_py = repo / 'scripts' / 'train.py'
s = train_py.read_text()
old = '''    model = timm.create_model(
        args.model, pretrained=not args.no_pretrained, num_classes=len(CLASSES)
    ).to(args.device)'''
new = '''    model = timm.create_model(
        args.model,
        pretrained=not args.no_pretrained,
        num_classes=len(CLASSES),
        img_size=args.image_size,
    ).to(args.device)'''
if old in s:
    train_py.write_text(s.replace(old, new))
    print('patched train.py')
elif 'img_size=args.image_size' in s:
    print('train.py already patched')
else:
    print('patch not applied; check train.py manually')


In [ ]:
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

SEARCH_ROOTS = (
    ["/content/drive/MyDrive", "/content"]
    if COLAB
    else ["/kaggle/input"]
)
print("search roots:", SEARCH_ROOTS)

In [ ]:
REQUIRED = {"ResearchImage", "Images_v1", "Images_v2"}


def find_dataset(roots):
    for root in roots:
        base = pathlib.Path(root)
        if not base.exists():
            continue
        for candidate in base.rglob("ResearchImage"):
            parent = candidate.parent
            if all((parent / name).is_dir() for name in REQUIRED):
                return parent
    raise FileNotFoundError(f"no dataset with {sorted(REQUIRED)} under {roots}")


DATASET_DIR = find_dataset(SEARCH_ROOTS)
print("dataset:", DATASET_DIR)

In [ ]:
target = repo / "data" / "dataset"
target.parent.mkdir(parents=True, exist_ok=True)
if target.exists() or target.is_symlink():
    if not target.is_symlink():
        raise SystemExit(f"{target} exists and is not a symlink; remove it first")
    target.unlink()
target.symlink_to(DATASET_DIR, target_is_directory=True)
print("linked:", target, "->", DATASET_DIR)

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements_cloud.txt")],
    check=True,
)
import torch

print("torch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected; switch the runtime to GPU before training")
print("device:", torch.cuda.get_device_name(0))

In [ ]:
ALL_MODELS = [
    {
        "model": "swin_tiny_patch4_window7_224.ms_in1k",
        "run_name": "swin_tiny_224_balanced",
        "image_size": 224,
        "batch_size": 16,
    },
    {
        "model": "convnext_tiny.fb_in1k",
        "run_name": "convnext_tiny_224_balanced",
        "image_size": 224,
        "batch_size": 16,
    },
    {
        "model": "densenet121.ra_in1k",
        "run_name": "densenet121_224_balanced",
        "image_size": 224,
        "batch_size": 32,
    },
]

# Choose one: "swin_tiny_224_balanced", "convnext_tiny_224_balanced", "densenet121_224_balanced"
ACTIVE_RUN_NAME = "swin_tiny_224_balanced"

MODELS = [m for m in ALL_MODELS if m["run_name"] == ACTIVE_RUN_NAME]
assert MODELS, f"Unknown ACTIVE_RUN_NAME: {ACTIVE_RUN_NAME}"

FOLDS = "0,1,2,3,4"
REPEATS = "0,1,2"
SEEDS = "42,43,44"
EPOCHS = 12
PATIENCE = 3
for m in MODELS:
    print(m["run_name"], m["model"], m["image_size"], m["batch_size"])
print(FOLDS, REPEATS, SEEDS, EPOCHS, PATIENCE)


In [ ]:
for cfg in MODELS:
    cmd = [
        sys.executable,
        str(repo / "scripts" / "train.py"),
        "--model", cfg["model"],
        "--run-name", cfg["run_name"],
        "--folds", FOLDS,
        "--repeats", REPEATS,
        "--seeds", SEEDS,
        "--epochs", str(EPOCHS),
        "--image-size", str(cfg["image_size"]),
        "--batch-size", str(cfg["batch_size"]),
        "--patience", str(PATIENCE),
        "--class-balanced",
        "--amp",
        "--resume",
        "--device", "cuda",
    ]
    print("\n===", cfg["run_name"], "===")
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=repo, check=True)


In [ ]:
for cfg in MODELS:
    subprocess.run(
        [sys.executable, str(repo / "scripts" / "summarize_finetuned.py"), "--run-name", cfg["run_name"]],
        cwd=repo,
        check=True,
    )


Outputs are written under `results/` in the repo. The last cell copies summary, confusion matrix, OOF file, and per-fold predictions for the selected model to persistent storage (`/content/drive/MyDrive/SkinDiseaseBD_runs` on Colab, `/kaggle/working/run_output` on Kaggle).


In [ ]:
out_dir = pathlib.Path(
    "/content/drive/MyDrive/SkinDiseaseBD_runs"
    if COLAB
    else "/kaggle/working/run_output"
)
out_dir.mkdir(parents=True, exist_ok=True)
copied = 0
for cfg in MODELS:
    run_name = cfg["run_name"]
    patterns = (
        f"finetuned/{run_name}_summary.json",
        f"finetuned/{run_name}_confusion_matrix.csv",
        f"predictions/finetuned_{run_name}_oof.csv",
        f"predictions/finetuned_{run_name}_fold*_test.csv",
    )
    for pattern in patterns:
        for path in (repo / "results").glob(pattern):
            shutil.copy2(path, out_dir / path.name)
            copied += 1
print(f"copied {copied} files to {out_dir}")
